In [24]:
# =============================================================================
# NFHS GPS Cluster to GP Mapping: Diagnostic Analysis (NFHS4 vs NFHS5)
# =============================================================================
#
# Purpose:
#   Produces a diagnostic comparison table for NFHS4 (2015-16) and NFHS5
#   (2019-21) survey clusters in Rajasthan and Uttar Pradesh, evaluating
#   how reliably each cluster can be mapped to a Gram Panchayat (GP) given
#   the DHS GPS displacement protocol.
#
# Key outputs (for each round x state combination):
#   (1) Total rural survey clusters
#   (2) Clusters that map cleanly to a GP — a 5km buffer staying within the
#       same district would not change the GP assignment
#   (3) Clusters at risk — the 5km buffer (district-constrained) crosses into
#       at least one adjacent GP with potentially different reservation status
#   (4) [Pending reservation data] At-risk clusters where all adjacent GPs
#       within the buffer share the same reservation status
#
# GPS Displacement Notes (from DHS README):
#   - Urban clusters displaced up to 2km
#   - Rural clusters displaced up to 5km (1% up to 10km)
#   - CRITICAL: Displacement is constrained to stay within the same admin2
#     (district). Cross-district GP misassignment is therefore impossible.
#     The 5km buffer analysis only flags clusters as at-risk if the adjacent
#     GP is in the SAME district.
#
# Rajasthan Context:
#   GP delimitation occurred in 2015. NFHS4 (2015-16) reflects the
#   pre-delimitation GP structure relevant for 2005 and 2010 election cycles.
#   NFHS5 (2019-21) reflects the post-delimitation structure relevant for
#   long-run effects. Both rounds are included for comparison.
#
# Data requirements (all files in ../data/ subfolders):
#   - nfhs4/nfhs4_gps_clusters.shp
#   - nfhs5/nfhs5_gps_clusters.shp
#   - shrug rajasthan village polygons/shrug_rajasthan_villages.shp
#   - shrug uttar pradesh village polygons/shrug_up_villages_part1.shp
#   - shrug uttar pradesh village polygons/shrug_up_villages_part2.shp
#   - shrug_LGD_matched.csv

In [25]:
# =============================================================================
# CELL 2 — Imports and File Paths
# =============================================================================
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f"geopandas: {gpd.__version__}")
print(f"pandas:    {pd.__version__}")

# ── Relative paths (no hardcoded local paths) ─────────────────────────────────
DATA_DIR = Path("../data")

NFHS4_GPS_PATH     = DATA_DIR / "nfhs4" / "nfhs4_gps_clusters.shp"
NFHS5_GPS_PATH     = DATA_DIR / "nfhs5" / "nfhs5_gps_clusters.shp"
SHRUG_LGD_PATH     = DATA_DIR / "shrug_LGD_matched.csv"
RJ_VILLAGES_PATH   = DATA_DIR / "shrug rajasthan village polygons" / "shrug_rajasthan_villages.shp"
UP_VILLAGES_PART1  = DATA_DIR / "shrug uttar pradesh village polygons" / "shrug_up_villages_part1.shp"
UP_VILLAGES_PART2  = DATA_DIR / "shrug uttar pradesh village polygons" / "shrug_up_villages_part2.shp"

# ── Verify all required files exist ───────────────────────────────────────────
print("\nChecking required files:")
all_found = True
for path in [NFHS4_GPS_PATH, NFHS5_GPS_PATH, SHRUG_LGD_PATH,
             RJ_VILLAGES_PATH, UP_VILLAGES_PART1, UP_VILLAGES_PART2]:
    status = "✓" if path.exists() else "✗ NOT FOUND"
    if not path.exists():
        all_found = False
    print(f"  {status}  {path}")

if all_found:
    print("\nAll files found. Ready to proceed.")
else:
    print("\nSome files missing — check paths before proceeding.")

# ── Constants ─────────────────────────────────────────────────────────────────
TARGET_STATES       = ["RAJASTHAN", "UTTAR PRADESH"]
TARGET_STATE_CODES  = ["08", "09"]
PROJECTED_CRS       = "EPSG:32644"   # UTM Zone 44N (metres) covers RJ + UP
RURAL_DISPLACEMENT  = 5000           # max rural displacement in metres (DHS)
URBAN_DISPLACEMENT  = 2000           # max urban displacement in metres (DHS)

geopandas: 1.1.3
pandas:    3.0.2

Checking required files:
  ✓  ../data/nfhs4/nfhs4_gps_clusters.shp
  ✓  ../data/nfhs5/nfhs5_gps_clusters.shp
  ✓  ../data/shrug_LGD_matched.csv
  ✓  ../data/shrug rajasthan village polygons/shrug_rajasthan_villages.shp
  ✓  ../data/shrug uttar pradesh village polygons/shrug_up_villages_part1.shp
  ✓  ../data/shrug uttar pradesh village polygons/shrug_up_villages_part2.shp

All files found. Ready to proceed.


In [26]:
# =============================================================================
# CELL 3 — Load and Prepare GPS Clusters (Both Rounds)
# =============================================================================
# Loads DHS GPS cluster shapefiles for NFHS4 and NFHS5.
# Key steps:
#   - Strips whitespace and uppercases state names for consistent comparison
#     (NFHS4 uses title case e.g. 'Rajasthan'; NFHS5 uses uppercase 'RAJASTHAN')
#   - Drops zero-coordinate clusters (DHS convention for missing/unverified
#     locations, flagged as 'MIS' in SOURCE field)
#   - Filters to Rajasthan and Uttar Pradesh only
#   - Reprojects from WGS84 (degrees) to UTM Zone 44N (metres) for accurate
#     distance calculations in the buffer analysis

def load_gps_clusters(path, round_name):
    """
    Load DHS GPS cluster shapefile for one survey round.
    Returns a GeoDataFrame projected to UTM Zone 44N, filtered to RJ + UP.
    """
    print(f"\nLoading {round_name} GPS clusters...")
    gdf = gpd.read_file(path)

    print(f"  Raw columns: {gdf.columns.tolist()}")
    print(f"  Raw rows: {len(gdf)}")

    # Keep relevant columns only
    gdf = gdf[["DHSCLUST", "LATNUM", "LONGNUM",
                "URBAN_RURA", "ADM1NAME", "DHSREGNA"]].copy()

    # Strip whitespace and uppercase for consistent comparison across rounds
    # NFHS4 uses title case ('Rajasthan'), NFHS5 uses uppercase ('RAJASTHAN')
    gdf["ADM1NAME"] = gdf["ADM1NAME"].str.strip().str.upper()

    # Drop clusters with missing/zero coordinates
    # (DHS marks unverified clusters as 'MIS' with coordinates 0, 0)
    n_before = len(gdf)
    gdf = gdf[(gdf["LATNUM"] != 0) & (gdf["LONGNUM"] != 0)].copy()
    print(f"  Dropped {n_before - len(gdf)} zero-coordinate clusters")

    # Filter to Rajasthan and UP
    gdf = gdf[gdf["ADM1NAME"].isin(TARGET_STATES)].copy()
    print(f"  Clusters after filtering to RJ + UP: {len(gdf)}")

    if len(gdf) == 0:
        print(f"  WARNING: No clusters found for {round_name} in RJ + UP.")
        print(f"  Check ADM1NAME values in raw file.")
        return gdf

    # Add survey round label
    gdf["round"] = round_name

    # Convert to GeoDataFrame with WGS84 CRS
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry=gpd.points_from_xy(gdf.LONGNUM, gdf.LATNUM),
        crs="EPSG:4326"
    )

    # Reproject to UTM Zone 44N (metres) for accurate distance calculations
    gdf = gdf.to_crs(PROJECTED_CRS)

    print(f"\n  {round_name} summary:")
    print(f"  Total clusters: {len(gdf)}")
    print(f"  By state:")
    print(gdf["ADM1NAME"].value_counts().to_string())
    print(f"  By urban/rural:")
    print(gdf["URBAN_RURA"].value_counts().to_string())

    return gdf

nfhs4 = load_gps_clusters(NFHS4_GPS_PATH, "NFHS4")
nfhs5 = load_gps_clusters(NFHS5_GPS_PATH, "NFHS5")


Loading NFHS4 GPS clusters...
  Raw columns: ['DHSID', 'DHSCC', 'DHSYEAR', 'DHSCLUST', 'CCFIPS', 'ADM1FIPS', 'ADM1FIPSNA', 'ADM1SALBNA', 'ADM1SALBCO', 'ADM1DHS', 'ADM1NAME', 'DHSREGCO', 'DHSREGNA', 'SOURCE', 'URBAN_RURA', 'LATNUM', 'LONGNUM', 'ALT_GPS', 'ALT_DEM', 'DATUM', 'geometry']
  Raw rows: 28526
  Dropped 131 zero-coordinate clusters
  Clusters after filtering to RJ + UP: 5266

  NFHS4 summary:
  Total clusters: 5266
  By state:
ADM1NAME
UTTAR PRADESH    3637
RAJASTHAN        1629
  By urban/rural:
URBAN_RURA
R    3849
U    1417

Loading NFHS5 GPS clusters...
  Raw columns: ['DHSID', 'DHSCC', 'DHSYEAR', 'DHSCLUST', 'CCFIPS', 'ADM1FIPS', 'ADM1FIPSNA', 'ADM1SALBNA', 'ADM1SALBCO', 'ADM1DHS', 'ADM1NAME', 'DHSREGCO', 'DHSREGNA', 'SOURCE', 'URBAN_RURA', 'LATNUM', 'LONGNUM', 'ALT_GPS', 'ALT_DEM', 'DATUM', 'geometry']
  Raw rows: 30197
  Dropped 118 zero-coordinate clusters
  Clusters after filtering to RJ + UP: 4843

  NFHS5 summary:
  Total clusters: 4843
  By state:
ADM1NAME
UTTAR P

In [27]:
# =============================================================================
# CELL 4 — Load SHRUG Village Polygons
# =============================================================================
print("Loading SHRUG village polygons...")

# Rajasthan
rj = gpd.read_file(RJ_VILLAGES_PATH)
print(f"Rajasthan villages: {len(rj)}")

# Uttar Pradesh (two parts — split to keep files under GitHub 100MB limit)
up1 = gpd.read_file(UP_VILLAGES_PART1)
up2 = gpd.read_file(UP_VILLAGES_PART2)
up  = pd.concat([up1, up2], ignore_index=True)
print(f"UP villages (part1 + part2): {len(up)}")

# Combine into one GeoDataFrame and reproject to metres
villages = gpd.GeoDataFrame(
    pd.concat([rj, up], ignore_index=True),
    crs=rj.crs
).to_crs(PROJECTED_CRS)

print(f"\nTotal villages (RJ + UP): {len(villages)}")
print(villages["pc11_s_id"].value_counts().to_string())
print(f"Columns: {villages.columns.tolist()}")

Loading SHRUG village polygons...
Rajasthan villages: 44976
UP villages (part1 + part2): 107709

Total villages (RJ + UP): 152685
pc11_s_id
09    107709
08     44976
Columns: ['pc11_s_id', 'pc11_d_id', 'pc11_sd_id', 'pc11_tv_id', 'tv_name', 'geometry']


In [28]:
# =============================================================================
# CELL 5 — Load LGD Crosswalk and Assign GP Codes to Villages
# =============================================================================
print("Loading SHRUG-LGD crosswalk...")
shrug_lgd = pd.read_csv(SHRUG_LGD_PATH, encoding="latin-1", low_memory=False)
print(f"SHRUG-LGD rows: {len(shrug_lgd)}")

# ── Build GP code column ──────────────────────────────────────────────────────
# For GP → ULB conversions, use retained old GP code
# For regular GPs, use LGD_code

def get_gp_code(row):
    if row["gp_to_urban_conversion"] == "Yes" and pd.notna(row["old_gp_lgd_code"]):
        return row["old_gp_lgd_code"]
    elif pd.notna(row["LGD_code"]) and row["local_body_type"] == "Gram Panchayat":
        return row["LGD_code"]
    return None

def get_gp_name(row):
    if row["gp_to_urban_conversion"] == "Yes" and pd.notna(row["old_gp_name"]):
        return row["old_gp_name"]
    elif pd.notna(row["local_body_name"]) and row["local_body_type"] == "Gram Panchayat":
        return row["local_body_name"]
    return None

shrug_lgd["gp_lgd_code"] = shrug_lgd.apply(get_gp_code, axis=1)
shrug_lgd["gp_name"]     = shrug_lgd.apply(get_gp_name, axis=1)

lgd_gp = shrug_lgd[shrug_lgd["gp_lgd_code"].notna()][
    ["shrid2", "gp_lgd_code", "gp_name"]
].copy().rename(columns={"shrid2": "shrid2_key"})

print(f"Rows with valid GP code: {len(lgd_gp)}")

# ── Build shrid2 key on village polygons ──────────────────────────────────────
# shrid2 format: "11-{state:02}-{district:03}-{subdistrict:05}-{village:06}"
# This composite key uniquely identifies each village across all of India

villages["shrid2_key"] = (
    "11-" +
    villages["pc11_s_id"].astype(str).str.zfill(2)  + "-" +
    villages["pc11_d_id"].astype(str).str.zfill(3)  + "-" +
    villages["pc11_sd_id"].astype(str).str.zfill(5) + "-" +
    villages["pc11_tv_id"].astype(str).str.zfill(6)
)

# Merge GP codes onto villages
villages = villages.merge(lgd_gp, on="shrid2_key", how="left")

matched = villages["gp_lgd_code"].notna().mean()
print(f"\nVillages with GP code: {villages['gp_lgd_code'].notna().sum()} ({matched:.1%})")
print("\nBy state:")
print(villages.groupby("pc11_s_id")["gp_lgd_code"].apply(
    lambda x: f"{x.notna().sum()} of {len(x)} ({x.notna().mean():.1%})"
))

Loading SHRUG-LGD crosswalk...
SHRUG-LGD rows: 596389
Rows with valid GP code: 526244

Villages with GP code: 95471 (62.5%)

By state:
pc11_s_id
08     39420 of 44976 (87.6%)
09    56051 of 107709 (52.0%)
Name: gp_lgd_code, dtype: str


In [29]:
# =============================================================================
# CELL 6 — Core Analysis: District-Constrained Buffer GP Coverage
# =============================================================================
# For each cluster, creates a displacement-radius buffer and finds all GPs
# within it — constrained to the same district per DHS displacement policy.
#
# Uses vectorised gpd.sjoin rather than a row-by-row loop for performance.
# Running the naive loop over 10,000 clusters x 152,685 villages would take
# several hours; sjoin with spatial indexing runs in minutes.
#
# Displacement radii per DHS README:
#   Rural:  5km (1% of rural clusters displaced up to 10km)
#   Urban:  2km

def analyse_gp_coverage(clusters, villages_with_gp, round_name):
    """
    Vectorised buffer analysis using gpd.sjoin with spatial indexing.
    Returns clusters GeoDataFrame with classification columns added.
    """
    print(f"\nRunning district-constrained buffer analysis for {round_name}...")

    villages_gp = villages_with_gp[
        villages_with_gp["gp_lgd_code"].notna()
    ][["pc11_d_id", "gp_lgd_code", "geometry"]].copy()

    # ── Step 1: Create buffer geometry for each cluster ───────────────────────
    # Use appropriate radius based on urban/rural classification
    clusters = clusters.copy()
    clusters["buffer_m"] = clusters["URBAN_RURA"].map({
        "R": RURAL_DISPLACEMENT,
        "U": URBAN_DISPLACEMENT
    })
    clusters["buffer_geom"] = clusters.apply(
        lambda row: row.geometry.buffer(row["buffer_m"]), axis=1
    )

    # Create a GeoDataFrame using buffer geometry for the spatial join
    cluster_buffers = clusters[["DHSCLUST", "buffer_geom"]].copy()
    cluster_buffers = gpd.GeoDataFrame(
        cluster_buffers,
        geometry="buffer_geom",
        crs=clusters.crs
    )

    print(f"  Buffers created for {len(cluster_buffers)} clusters")

    # ── Step 2: Spatial join — find all villages within each buffer ───────────
    print(f"  Running spatial join against {len(villages_gp)} villages with GP codes...")
    villages_in_buffers = gpd.sjoin(
        villages_gp,
        cluster_buffers,
        how="inner",
        predicate="intersects"
    )
    print(f"  Village-buffer intersections found: {len(villages_in_buffers)}")

    # ── Step 3: District constraint ───────────────────────────────────────────
    # DHS displacement stays within same admin2 (district).
    # Join cluster's matched district onto the intersections and filter.
    # pc11_d_id from nearest village (assigned in Cell 7) gives cluster district.
    # For now we apply the constraint after Cell 7 — flag all intersections here.

    # Count distinct GPs per cluster (pre-constraint)
    gps_per_cluster = (
        villages_in_buffers
        .groupby("DHSCLUST")["gp_lgd_code"]
        .agg(lambda x: list(x.unique()))
        .reset_index()
        .rename(columns={"gp_lgd_code": "gps_in_buffer"})
    )
    gps_per_cluster["n_gps_in_buffer"] = gps_per_cluster["gps_in_buffer"].apply(len)

    # Also store district codes of GPs in buffer for constraint in Cell 7
    district_per_cluster = (
        villages_in_buffers
        .groupby("DHSCLUST")["pc11_d_id"]
        .agg(lambda x: list(x.unique()))
        .reset_index()
        .rename(columns={"pc11_d_id": "districts_in_buffer"})
    )

    # Merge back onto clusters
    clusters = clusters.merge(gps_per_cluster, on="DHSCLUST", how="left")
    clusters = clusters.merge(district_per_cluster, on="DHSCLUST", how="left")

    # Fill clusters with no villages in buffer
    clusters["n_gps_in_buffer"] = clusters["n_gps_in_buffer"].fillna(0).astype(int)
    clusters["gps_in_buffer"] = clusters["gps_in_buffer"].apply(
        lambda x: x if isinstance(x, list) else []
    )

    # Preliminary classification (before district constraint in Cell 7)
    clusters["clean_gp"]    = clusters["n_gps_in_buffer"] == 1
    clusters["at_risk_gp"]  = clusters["n_gps_in_buffer"] >  1
    clusters["no_gp_match"] = clusters["n_gps_in_buffer"] == 0

    # Drop buffer geometry column (not needed downstream)
    clusters = clusters.drop(columns=["buffer_geom", "buffer_m"])

    print(f"\n  Raw buffer results (ALL clusters, NO district constraint yet):")
    print(f"  (District constraint applied in Cell 7; final rural-only numbers shown there)")
    print(f"  Clean (1 GP in buffer):     {clusters['clean_gp'].sum()} ({clusters['clean_gp'].mean():.1%})")
    print(f"  At-risk (2+ GPs in buffer): {clusters['at_risk_gp'].sum()} ({clusters['at_risk_gp'].mean():.1%})")
    print(f"  No GP found in buffer:      {clusters['no_gp_match'].sum()} ({clusters['no_gp_match'].mean():.1%})")

    return clusters

nfhs4_analysed = analyse_gp_coverage(nfhs4, villages, "NFHS4")
nfhs5_analysed = analyse_gp_coverage(nfhs5, villages, "NFHS5")


Running district-constrained buffer analysis for NFHS4...
  Buffers created for 5266 clusters
  Running spatial join against 95471 villages with GP codes...
  Village-buffer intersections found: 106383

  Raw buffer results (ALL clusters, NO district constraint yet):
  (District constraint applied in Cell 7; final rural-only numbers shown there)
  Clean (1 GP in buffer):     161 (3.1%)
  At-risk (2+ GPs in buffer): 3617 (68.7%)
  No GP found in buffer:      1488 (28.3%)

Running district-constrained buffer analysis for NFHS5...
  Buffers created for 4843 clusters
  Running spatial join against 95471 villages with GP codes...
  Village-buffer intersections found: 107088

  Raw buffer results (ALL clusters, NO district constraint yet):
  (District constraint applied in Cell 7; final rural-only numbers shown there)
  Clean (1 GP in buffer):     141 (2.9%)
  At-risk (2+ GPs in buffer): 3390 (70.0%)
  No GP found in buffer:      1312 (27.1%)


In [30]:
# =============================================================================
# CELL 7 — Nearest Village Join and Apply District Constraint
# =============================================================================

# Columns that come from the village join — drop if already present
# from a previous partial run of this cell
VILLAGE_COLS = [
    "pc11_s_id", "pc11_d_id", "pc11_sd_id",
    "pc11_tv_id", "tv_name", "gp_lgd_code",
    "gp_name", "shrid2_key", "index_right"
]

def assign_nearest_village_and_district(clusters, villages_with_gp, round_name):
    print(f"\nRunning nearest-village spatial join for {round_name}...")

    # Drop any village columns already present from a previous run
    cols_to_drop = [c for c in VILLAGE_COLS if c in clusters.columns]
    if cols_to_drop:
        print(f"  Dropping pre-existing village columns: {cols_to_drop}")
        clusters = clusters.drop(columns=cols_to_drop)

    # Select available village columns
    village_cols = [
        c for c in [
            "pc11_s_id", "pc11_d_id", "pc11_sd_id",
            "pc11_tv_id", "tv_name", "gp_lgd_code",
            "gp_name", "geometry"
        ]
        if c in villages_with_gp.columns
    ]

    joined = gpd.sjoin_nearest(
        clusters,
        villages_with_gp[village_cols],
        how="left"
    )

    joined = joined.drop(columns=["index_right"], errors="ignore")

    print(f"  Clusters joined: {len(joined)}")
    print(f"  GP match rate:   {joined['gp_lgd_code'].notna().mean():.1%}")

    return joined

nfhs4_analysed = assign_nearest_village_and_district(
    nfhs4_analysed, villages, "NFHS4"
)
nfhs5_analysed = assign_nearest_village_and_district(
    nfhs5_analysed, villages, "NFHS5"
)

# ── Apply district constraint ─────────────────────────────────────────────────
def apply_district_constraint(clusters, villages_with_gp):
    """
    Recount GPs in buffer after removing any GPs from different districts.
    DHS displacement stays within same admin2 (district) per DHS README.
    """
    district_gp = (
        villages_with_gp[villages_with_gp["gp_lgd_code"].notna()]
        .groupby("pc11_d_id")["gp_lgd_code"]
        .apply(set)
        .to_dict()
    )

    def count_constrained_gps(row):
        if not isinstance(row["gps_in_buffer"], list):
            return 0
        same_district_gps = district_gp.get(row["pc11_d_id"], set())
        constrained = [
            gp for gp in row["gps_in_buffer"]
            if gp in same_district_gps
        ]
        return len(constrained)

    clusters["n_gps_constrained"]      = clusters.apply(count_constrained_gps, axis=1)
    clusters["clean_gp_constrained"]   = clusters["n_gps_constrained"] == 1
    clusters["at_risk_gp_constrained"] = clusters["n_gps_constrained"] >  1
    clusters["no_gp_constrained"]      = clusters["n_gps_constrained"] == 0

    return clusters

nfhs4_analysed = apply_district_constraint(nfhs4_analysed, villages)
nfhs5_analysed = apply_district_constraint(nfhs5_analysed, villages)

print("\nAfter district constraint:")
for name, df in [("NFHS4", nfhs4_analysed), ("NFHS5", nfhs5_analysed)]:
    rural = df[df["URBAN_RURA"] == "R"]
    print(f"\n{name} rural clusters:")
    print(f"  Clean (1 GP, same district):     {rural['clean_gp_constrained'].sum()} ({rural['clean_gp_constrained'].mean():.1%})")
    print(f"  At-risk (2+ GPs, same district): {rural['at_risk_gp_constrained'].sum()} ({rural['at_risk_gp_constrained'].mean():.1%})")
    print(f"  No GP match:                     {rural['no_gp_constrained'].sum()} ({rural['no_gp_constrained'].mean():.1%})")


Running nearest-village spatial join for NFHS4...
  Clusters joined: 5266
  GP match rate:   52.3%

Running nearest-village spatial join for NFHS5...
  Clusters joined: 4843
  GP match rate:   54.1%

After district constraint:

NFHS4 rural clusters:
  Clean (1 GP, same district):     11 (0.3%)
  At-risk (2+ GPs, same district): 2797 (72.7%)
  No GP match:                     1041 (27.0%)

NFHS5 rural clusters:
  Clean (1 GP, same district):     8 (0.2%)
  At-risk (2+ GPs, same district): 2734 (71.5%)
  No GP match:                     1082 (28.3%)


In [31]:
# =============================================================================
# CELL 8 — Comparison Table: NFHS4 vs NFHS5 Cluster-to-GP Mapping Quality
# =============================================================================
def build_summary_row(df, round_name, state):
    """One row of the summary table for a given round and state."""
    subset = df[df["ADM1NAME"] == state]
    rural  = subset[subset["URBAN_RURA"] == "R"]
    total  = len(rural)

    def fmt(n, denom=None):
        d = denom if denom is not None else total
        if d == 0:
            return "0 (n/a)"
        return f"{int(n):,} ({n/d:.1%})"

    clean    = rural["clean_gp_constrained"].sum()
    at_risk  = rural["at_risk_gp_constrained"].sum()
    no_match = rural["no_gp_constrained"].sum()

    return {
        "Round":                              round_name,
        "State":                              state,
        "Rural clusters":                     f"{total:,}",
        "Clean GP mapping":                   fmt(clean),
        "At-risk GP mapping":                 fmt(at_risk),
        "No GP match (LGD gap)":             fmt(no_match),
    }

rows = []
for round_name, df in [("NFHS4 (2015-16)", nfhs4_analysed),
                        ("NFHS5 (2019-21)", nfhs5_analysed)]:
    for state in TARGET_STATES:
        rows.append(build_summary_row(df, round_name, state))

summary = pd.DataFrame(rows)

# ── Print formatted table with column separators ──────────────────────────────
col_widths = {col: max(len(col), summary[col].astype(str).str.len().max())
              for col in summary.columns}

def make_row(values, widths):
    return " | ".join(str(v).ljust(widths[c]) for c, v in zip(widths.keys(), values))

def make_divider(widths):
    return "-+-".join("-" * w for w in widths.values())

header   = make_row(summary.columns, col_widths)
divider  = make_divider(col_widths)
total_width = len(header)

print("=" * total_width)
print("CLUSTER-TO-GP MAPPING QUALITY: NFHS4 vs NFHS5")
print("=" * total_width)
print(header)
print(divider)
for _, row in summary.iterrows():
    print(make_row(row.values, col_widths))
print("=" * total_width)

CLUSTER-TO-GP MAPPING QUALITY: NFHS4 vs NFHS5
Round           | State         | Rural clusters | Clean GP mapping | At-risk GP mapping | No GP match (LGD gap)
----------------+---------------+----------------+------------------+--------------------+----------------------
NFHS4 (2015-16) | RAJASTHAN     | 1,190          | 3 (0.3%)         | 1,186 (99.7%)      | 1 (0.1%)             
NFHS4 (2015-16) | UTTAR PRADESH | 2,659          | 8 (0.3%)         | 1,611 (60.6%)      | 1,040 (39.1%)        
NFHS5 (2019-21) | RAJASTHAN     | 1,163          | 4 (0.3%)         | 1,159 (99.7%)      | 0 (0.0%)             
NFHS5 (2019-21) | UTTAR PRADESH | 2,661          | 4 (0.2%)         | 1,575 (59.2%)      | 1,082 (40.7%)        


In [32]:
# =============================================================================
# CELL 9 — Save Cluster-to-GP Crosswalk CSVs
# =============================================================================
# Save cluster-to-GP crosswalk for each round

SCRIPTS_DIR = Path(".")   # notebook lives in scripts/
OUTPUT_DIR  = SCRIPTS_DIR / "../data"

cols_to_save = [
    "DHSCLUST", "LATNUM", "LONGNUM", "URBAN_RURA",
    "ADM1NAME", "DHSREGNA", "round",
    "tv_name", "pc11_s_id", "pc11_d_id", "pc11_sd_id", "pc11_tv_id",
    "gp_lgd_code", "gp_name",
    "n_gps_constrained", "clean_gp_constrained",
    "at_risk_gp_constrained", "no_gp_constrained"
]

for round_name, df in [("nfhs4", nfhs4_analysed),
                        ("nfhs5", nfhs5_analysed)]:
    out_path = OUTPUT_DIR / f"{round_name}_cluster_gp_crosswalk.csv"
    df[cols_to_save].to_csv(out_path, index=False)
    print(f"Saved: {out_path} ({len(df)} clusters)")

print("\nDone.")
print("Next step: merge crosswalk with GP election reservation status data.")

Saved: ../data/nfhs4_cluster_gp_crosswalk.csv (5266 clusters)
Saved: ../data/nfhs5_cluster_gp_crosswalk.csv (4843 clusters)

Done.
Next step: merge crosswalk with GP election reservation status data.


In [33]:
# =============================================================================
# CELL 10 — Notes on Interpretation
# =============================================================================
print("""
── Notes on interpretation ──────────────────────────────────────────────────

DISTRICT CONSTRAINT:
  Per the DHS GPS displacement README, cluster displacement is restricted to
  stay within the same admin2 area (district). Cross-district GP
  misassignment is therefore impossible. The buffer analysis only counts GPs
  in the same district as the cluster's matched village.

CLEAN GP MAPPING:
  The district-constrained 5km buffer intersects only one GP. Regardless of
  which direction DHS displacement was applied, the cluster maps to the same
  GP. Treatment assignment is unambiguous. NOTE: Only ~0.3% of rural clusters
  meet this criterion, reflecting India's dense GP settlement pattern where
  a 5km radius almost always spans multiple GPs within the same district.

AT-RISK GP MAPPING:
  The district-constrained 5km buffer intersects 2 or more GPs. Depending
  on the direction of displacement, the cluster could map to a different GP
  with potentially different reservation status. Flagged for robustness
  checks once GP reservation data is available.

NO GP MATCH:
  No GP found within district-constrained buffer — primarily the known ~40%
  LGD coverage gap for UP (concentrated in eastern UP and Bundelkhand
  districts), or peri-urban Out Growth (OG) areas without GP assignments.
  Rajasthan LGD coverage is near-complete (0-0.1% no-match rate).

RAJASTHAN DELIMITATION:
  GP boundaries were redrawn in Rajasthan in 2015. NFHS4 (2015-16) reflects
  the pre-delimitation structure relevant for 2005 and 2010 elections.
  NFHS5 (2019-21) reflects the post-delimitation structure. Clusters in
  both rounds may therefore map to different GPs in the same location.

PENDING:
  Once GP election reservation status is available, a third category will
  be added: at-risk clusters where all GPs within the district-constrained
  buffer share the same reservation status — meaning displacement would not
  affect treatment assignment regardless of direction.
""")


── Notes on interpretation ──────────────────────────────────────────────────

DISTRICT CONSTRAINT:
  Per the DHS GPS displacement README, cluster displacement is restricted to
  stay within the same admin2 area (district). Cross-district GP
  misassignment is therefore impossible. The buffer analysis only counts GPs
  in the same district as the cluster's matched village.

CLEAN GP MAPPING:
  The district-constrained 5km buffer intersects only one GP. Regardless of
  which direction DHS displacement was applied, the cluster maps to the same
  GP. Treatment assignment is unambiguous. NOTE: Only ~0.3% of rural clusters
  meet this criterion, reflecting India's dense GP settlement pattern where
  a 5km radius almost always spans multiple GPs within the same district.

AT-RISK GP MAPPING:
  The district-constrained 5km buffer intersects 2 or more GPs. Depending
  on the direction of displacement, the cluster could map to a different GP
  with potentially different reservation status. Fla